Core NLP Applications and Multilingual Models
You are a lead developer building an AI-powered knowledge
management platform for a large corporation. The platform needs two
critical capabilities:
● Extracting precise answers from long documents based on user
questions (Question Answering).
● Generating concise summaries of meeting transcripts or legal texts
(Text Summarization).
Furthermore, the system must be globally viable, specifically supporting
key documents written in common Indian languages. Your task is to
implement and benchmark these two core applications using modern
Transformer models.

Tasks:
Part A: Extractive Question Answering (QA)
Task A.1: Implementing QA Pipeline Extractive QA involves
predicting the start and end tokens of the answer within a given context.
1. Load a QA-specific model and tokenizer (e.g.,
AutoModelForQuestionAnswering and a corresponding
fine-tuned BERT/DistilBERT model).
2. Define a sample Context (a paragraph of text) and a Question.
3. Tokenize the input (Question + Context) and find the token
indices corresponding to the [CLS] and [SEP] tokens.
4. Perform the model inference to get the start logits and end logits.

5. Use argmax on the logits to find the predicted start position and
end position of the answer span.
6. Decode the token span to present the final extracted answer text.

Task A.2: QA Output and Analysis
1. Test your QA pipeline with at least two different (Context,
Question) pairs.
2. Analysis: Briefly explain why the QA task is classified as an
Extractive task and how the BERT architecture (specifically its
token-level output) is inherently suited to predicting the start and
end indices.

Part B: Abstractive Text Summarization
Task B.3: Implementing Summarization Pipeline Abstractive
summarization uses a Seq2Seq (Encoder-Decoder) model to generate a
new summary, not just extract sentences.
1. Load a Seq2Seq model for summarization (e.g., T5 or BART) and
its tokenizer.
2. Define a sample Long Document (the source text).
3. Use the model's generate() method, setting parameters such as
max_length, min_length, and num_beams (for beam search
decoding).
4. Generate and print the abstractive summary.
Task B.4: Evaluation using ROUGE Score The provided notebook has
set up the necessary tools for ROUGE evaluation.
1. Provide the model's generated summary (from Task B.3) as the
System Summary.
2. Provide a Human-written Reference Summary for the same
document.
3. Use the evaluate library and the ROUGE metric to calculate the
ROUGE-1, ROUGE-2, and ROUGE-L scores (F1-scores are
sufficient).
4. Analysis: Interpret the meaning of the ROUGE-L score (Longest
Common Subsequence). Why is ROUGE a better metric for
generation tasks than simple token accuracy?

Part C: Multilingual Extension (Focus on Indian Languages)
Task C.5: Model Proposal and Application Address the need for
multilingual support.
1. Proposal: Identify and name one popular Multilingual Large
Language Model (MLLM) (e.g., XLM-RoBERTa, mBART, or a
specific Indic-focused model) suitable for processing Indian
languages.
2. Description: Explain (in a brief paragraph) the core architectural
feature of this model that allows it to handle multiple languages

efficiently, typically by sharing a common, large multilingual
vocabulary and joint training across many languages.
Deliverables
1. A fully executed Python Notebook (.ipynb) demonstrating the
Extractive QA (Task A.1) and the Abstractive Summarization
with ROUGE Evaluation (Task B.3 & B.4).
2. Analysis Report: A complete written answer for the three
analysis sections (A.2, B.4, C.5).

In [1]:
pip install transformers torch

In [2]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

# 1. Load a QA-specific model and tokenizer:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased-distilled-squad")
model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-cased-distilled-squad")

print("Model and Tokenizer loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Model and Tokenizer loaded successfully!


In [7]:
# 2. Define a sample Context and a Question:
context = r"""Hugging Face Inc. is an American company that develops tools for building applications using machine learning. It is most known for its Transformers library, built for vision and natural language processing applications and its web platform that allows users to share and deploy machine learning models. In 2021, the company was valued at $2 billion. In 2022, Hugging Face announced a partnership with Amazon Web Services (AWS) to make its products available to AWS customers and provide training and support.
"""

question = "What is Hugging Face most known for?"

print(f"Question: {question}")
print(f"Context: {context[:100]}...")

Question: What is Hugging Face most known for?
Context: Hugging Face Inc. is an American company that develops tools for building applications using machine...


In [8]:
# 3. Tokenize the input:
inputs = tokenizer(question, context, return_tensors="pt")

cls_token_index = (inputs["input_ids"] == tokenizer.cls_token_id).nonzero(as_tuple=True)[1].item()
sep_token_indices = (inputs["input_ids"] == tokenizer.sep_token_id).nonzero(as_tuple=True)[1]
sep_token_index_question = sep_token_indices[0].item() # SEP after question
sep_token_index_context = sep_token_indices[1].item() # SEP after context

print(f"Input IDs length: {len(inputs['input_ids'][0])}")
print(f"[CLS] token index: {cls_token_index}")
print(f"[SEP] token index after question: {sep_token_index_question}")
print(f"[SEP] token index after context: {sep_token_index_context}")


Input IDs length: 109
[CLS] token index: 0
[SEP] token index after question: 10
[SEP] token index after context: 108


In [9]:
# 4. Perform the model inference to get the start logits and end logits:
with torch.no_grad():
    outputs = model(**inputs)

start_logits = outputs.start_logits
end_logits = outputs.end_logits

print(f"Start logits shape: {start_logits.shape}")
print(f"End logits shape: {end_logits.shape}")

Start logits shape: torch.Size([1, 109])
End logits shape: torch.Size([1, 109])


In [10]:
# 5. Use argmax on the logits to find the predicted start position and end position of the answer span.
answer_start_index = torch.argmax(start_logits)
answer_end_index = torch.argmax(end_logits)

print(f"Predicted start token index: {answer_start_index.item()}")
print(f"Predicted end token index: {answer_end_index.item()}")

Predicted start token index: 36
Predicted end token index: 37


In [11]:
# 6. Decode the token span to present the final extracted answer text:
predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
extracted_answer = tokenizer.decode(predict_answer_tokens, skip_special_tokens=True)

print(f"Extracted Answer: {extracted_answer}")

Extracted Answer: Transformers library


In [12]:
# Define a second sample Context and Question
context_2 = r"""The Amazon rainforest is the largest rainforest in the world, covering an area of about 5.5 million square kilometers. It is home to an incredible diversity of plant and animal life, including jaguars, sloths, and countless species of insects. The rainforest plays a crucial role in regulating the Earth's climate by absorbing vast amounts of carbon dioxide and releasing oxygen.
"""

question_2 = "What is the Amazon rainforest known for?"

print(f"Question 2: {question_2}")
print(f"Context 2: {context_2[:100]}...")

Question 2: What is the Amazon rainforest known for?
Context 2: The Amazon rainforest is the largest rainforest in the world, covering an area of about 5.5 million ...


In [13]:
# Tokenize the input for the second pair
inputs_2 = tokenizer(question_2, context_2, return_tensors="pt")

# Perform inference for the second pair
with torch.no_grad():
    outputs_2 = model(**inputs_2)

start_logits_2 = outputs_2.start_logits
end_logits_2 = outputs_2.end_logits

# Find predicted start and end positions
answer_start_index_2 = torch.argmax(start_logits_2)
answer_end_index_2 = torch.argmax(end_logits_2)

# Decode the token span to present the final extracted answer text
predict_answer_tokens_2 = inputs_2.input_ids[0, answer_start_index_2 : answer_end_index_2 + 1]
extracted_answer_2 = tokenizer.decode(predict_answer_tokens_2, skip_special_tokens=True)

print(f"Extracted Answer 2: {extracted_answer_2}")

Extracted Answer 2: largest rainforest in the world


### Task A.2: Analysis

**Why is Question Answering an Extractive task?**

Extractive Question Answering means the model finds the answer directly within the text you provide. It doesn't create new sentences or rephrase information. Instead, it identifies the exact start and end points of the answer span from the original document. Think of it like highlighting the answer in a book – the answer is already there, you just need to find its location.

**How is BERT suited for predicting start and end indices?**

BERT (and similar models) is great for this because it processes text by giving a unique meaning (an 'embedding') to each word, considering its context. For QA, BERT is trained to predict two things for every word in the document: the probability that it's the *start* of the answer, and the probability that it's the *end* of the answer. It learns to do this effectively because its internal 'attention' mechanism helps it understand how each word relates to the question and other words in the document, allowing it to pinpoint the correct answer boundaries very accurately.

### Part B: Abstractive Text Summarization
#### Task B.3: Implementing Summarization Pipeline

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load a Seq2Seq model for summarization (e.g., T5) and its tokenizer
# We'll use 't5-small' for demonstration purposes as it's quicker to download and run.
tokenizer_summ = AutoTokenizer.from_pretrained("t5-small")
model_summ = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

print("Summarization Model and Tokenizer loaded successfully!")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Summarization Model and Tokenizer loaded successfully!


In [15]:
# 2. Define a sample Long Document (the source text)
long_document = r"""The rapid advancement of artificial intelligence (AI) has sparked both excitement and concern across various sectors. AI, particularly machine learning, is being deployed in fields ranging from healthcare and finance to transportation and entertainment. In healthcare, AI assists in diagnosing diseases, personalizing treatment plans, and accelerating drug discovery. Financial institutions leverage AI for fraud detection, algorithmic trading, and customer service. Autonomous vehicles, powered by AI, promise to revolutionize transportation, though safety and ethical considerations remain paramount.

However, the proliferation of AI also raises important questions about job displacement, algorithmic bias, and privacy. As AI systems become more sophisticated, they have the potential to automate tasks traditionally performed by humans, leading to concerns about the future of work. Furthermore, if AI models are trained on biased data, they can perpetuate and even amplify existing societal inequalities. Ensuring the ethical development and deployment of AI is a critical challenge, requiring collaboration between researchers, policymakers, and industry leaders to establish robust regulatory frameworks and promote responsible innovation.
"""

print("Sample long document defined.")

Sample long document defined.


In [16]:
# Prepend the input with the task prefix for T5 models
input_text = "summarize: " + long_document

# Tokenize the input document
inputs_summ = tokenizer_summ(input_text, return_tensors="pt", max_length=512, truncation=True)

# 3. Use the model's generate() method
# Setting parameters such as max_length, min_length, and num_beams (for beam search decoding)
summary_ids = model_summ.generate(
    inputs_summ["input_ids"],
    max_length=150, # Maximum length of the generated summary
    min_length=40,  # Minimum length of the generated summary
    num_beams=4,    # Number of beams for beam search. Higher values generally lead to better summaries but take longer.
    early_stopping=True # Stop beam search when all beams have finished
)

# Decode the generated summary
summary = tokenizer_summ.decode(summary_ids[0], skip_special_tokens=True)

# 4. Generate and print the abstractive summary
print("\nGenerated Abstractive Summary:")
print(summary)


Generated Abstractive Summary:
the rapid advancement of artificial intelligence (AI) has sparked both excitement and concern across various sectors. in healthcare, AI assists in diagnosing diseases, personalizing treatment plans, and accelerating drug discovery. autonomous vehicles, powered by AI, promise to revolutionize transportation.


#### Task B.4: Evaluation using ROUGE Score

In [17]:
pip install evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=a43b6d2e8a9844f2ebd8c0e32fd52de3ec8a794ac365f20a035d368daac95fc0
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [18]:
import evaluate

# 1. Provide the model's generated summary (from Task B.3) as the System Summary.
system_summary = summary # 'summary' variable should contain the output from Task B.3

# 2. Provide a Human-written Reference Summary for the same document.
# This reference summary should be for the 'long_document' defined in Task B.3
reference_summary = "AI is rapidly advancing and used in healthcare, finance, and transportation, leading to excitement and concerns. It aids in disease diagnosis, fraud detection, and autonomous vehicles. However, AI raises issues like job displacement, algorithmic bias, and privacy, necessitating ethical development and regulation."

print("System Summary:", system_summary)
print("Reference Summary:", reference_summary)

System Summary: the rapid advancement of artificial intelligence (AI) has sparked both excitement and concern across various sectors. in healthcare, AI assists in diagnosing diseases, personalizing treatment plans, and accelerating drug discovery. autonomous vehicles, powered by AI, promise to revolutionize transportation.
Reference Summary: AI is rapidly advancing and used in healthcare, finance, and transportation, leading to excitement and concerns. It aids in disease diagnosis, fraud detection, and autonomous vehicles. However, AI raises issues like job displacement, algorithmic bias, and privacy, necessitating ethical development and regulation.


In [19]:
# 3. Use the evaluate library and the ROUGE metric to calculate the ROUGE-1, ROUGE-2, and ROUGE-L scores (F1-scores are sufficient).
rouge = evaluate.load("rouge")

results = rouge.compute(predictions=[system_summary], references=[reference_summary])

print("\nROUGE Scores (F1-scores):")
for key, value in results.items():
    print(f"{key}: {value:.4f}")


ROUGE Scores (F1-scores):
rouge1: 0.2963
rouge2: 0.0759
rougeL: 0.2222
rougeLsum: 0.2222


#### Task B.4: Analysis

**Interpret the meaning of the ROUGE-L score (Longest Common Subsequence).**

ROUGE-L measures the F1-score based on the Longest Common Subsequence (LCS) between the generated summary and the reference summary. The LCS considers the longest sequence of words that appear in both the generated and reference summaries, in the same order, but not necessarily contiguously. It captures the main information flow and sentence-level structure more effectively than unigram or bigram overlap, giving a sense of how well the generated summary reproduces the significant phrases and overall structure of the reference.

**Why is ROUGE a better metric for generation tasks than simple token accuracy?**

ROUGE is superior to simple token accuracy for generation tasks like summarization because:

1.  **Flexibility in Phrasing:** Generated text can convey the same meaning using different words or sentence structures. Simple token accuracy would penalize these variations even if the meaning is preserved. ROUGE, especially ROUGE-1 and ROUGE-2, allows for some lexical variation while still measuring content overlap.
2.  **Captures Semantic Overlap (to an extent):** By measuring overlapping n-grams (words or sequences of words), ROUGE provides a better proxy for semantic similarity and content coverage than just individual token matches. For example, if a reference says "car accident" and a model generates "automobile crash," token accuracy might be low, but ROUGE-1 would still capture some overlap.
3.  **Addresses Fluency and Cohesion (ROUGE-L):** ROUGE-L's focus on the longest common subsequence helps assess how well the generated summary maintains the flow and structure of the reference, which is crucial for readability and coherence in generated text.
4.  **Reference-Based Evaluation:** Generation tasks often have multiple valid outputs. ROUGE compares the generated text against one or more human-written references, acknowledging that there isn't a single 'correct' answer, unlike classification where token accuracy is more appropriate for exact matches.

### Part C: Multilingual Extension (Focus on Indian Languages)
#### Task C.5: Model Proposal and Application

**1. Proposal: Multilingual Large Language Model (MLLM)**

For processing Indian languages efficiently, a highly suitable model would be **XLM-RoBERTa (Cross-lingual Language Model RoBERTa)**.

**2. Description: Core Architectural Features**

XLM-RoBERTa's effectiveness in handling multiple languages stems primarily from two core architectural features: a **shared, large multilingual vocabulary** and **joint training across many languages**. It uses a single SentencePiece vocabulary that encompasses characters and subword units from over 100 languages, including many Indian languages. This allows it to represent words from different languages within the same embedding space. During pre-training, XLM-RoBERTa is trained on massive amounts of text data from all these languages simultaneously using a Masked Language Model objective. This joint training forces the model to learn universal linguistic patterns and shared semantic representations across languages, rather than treating each language as a separate entity. As a result, knowledge gained from one language can be transferred and applied to others, making it highly effective for cross-lingual tasks and low-resource languages like many in India, even without explicit parallel data for fine-tuning.